# SAiDL Summer Assignment 2026 — Core ML

**Run cells top-to-bottom.** Each experiment cell is independent after setup.

Experiment plan:
1. Setup (run once)
2. Baseline: Vanilla MHA + Sinusoidal
3. Attention variants (Sliding Window, GQA, ReLU, Sparse) + Sinusoidal
4. Positional variants (RoPE, ALiBi, Relative) + Vanilla MHA
5. Hybrid models (Conv-Before-Attn, Gated Conv FFN, Interleaved)
6. Benchmark all checkpoints
7. Download results

In [ ]:
# ══ 0A  Clone repo and install packages (run once) ═══════════════════════════
import os, sys

REPO = '/content/SAiDL-Summer-Assignment-2026'

if not os.path.exists(REPO):
    !git clone https://github.com/VvS-2403/SAiDL-Summer-Assignment-2026.git {REPO}
else:
    !cd {REPO} && git pull --quiet

%cd {REPO}
sys.path.insert(0, REPO)

!pip install -q torch transformers datasets wandb hydra-core omegaconf \
    einops tqdm scikit-learn umap-learn matplotlib numpy

print('✅ Setup complete')

In [ ]:
# ══ 0B  W&B login ════════════════════════════════════════════════════════════
import wandb
wandb.login()   # paste your API key from https://wandb.ai/authorize

In [ ]:
# ══ 0C  Experiment runner helper ═════════════════════════════════════════════
import subprocess, sys, os
REPO = '/content/SAiDL-Summer-Assignment-2026'

def run_experiment(attn, pos, model='transformer', extra=None):
    """
    Run one training experiment.

    attn  : 'vanilla' | 'sliding_window' | 'gqa' | 'relu' | 'sparse'
    pos   : 'sinusoidal' | 'rope' | 'alibi' | 'relative'
    model : 'transformer' | 'hybrid'
    extra : list of additional Hydra overrides, e.g. ['model.hybrid.type=gated_conv_ffn']
    """
    name = f'{attn}_{pos}_{model}'
    print(f'\n{"="*60}\n  Starting: {name}\n{"="*60}')

    cmd = [
        sys.executable,
        f'{REPO}/core_ml/train/train.py',
        f'attention={attn}',
        f'positional={pos}',
        f'model={model}',
        'dataset.batch_size=8',
        'dataset.num_workers=2',
        f'experiment_name={name}',
    ]
    if extra:
        cmd += extra

    result = subprocess.run(cmd, cwd=REPO)
    status = '✅ DONE' if result.returncode == 0 else '❌ FAILED'
    print(f'{status}: {name}')
    return result.returncode

print('Helper ready.')

---
## Part 1 — Baseline

In [ ]:
# ══ 1.1  Baseline: Vanilla MHA + Sinusoidal ══════════════════════════════════
run_experiment('vanilla', 'sinusoidal')

---
## Part 2 — Attention Variants (all with Sinusoidal PE)

In [ ]:
# ══ 2.1  Sliding Window Attention ════════════════════════════════════════════
run_experiment('sliding_window', 'sinusoidal')

In [ ]:
# ══ 2.2  Grouped Query Attention (GQA) ═══════════════════════════════════════
run_experiment('gqa', 'sinusoidal')

In [ ]:
# ══ 2.3  ReLU (Softmax-free) Attention ═══════════════════════════════════════
run_experiment('relu', 'sinusoidal')

In [ ]:
# ══ 2.4  Sparse Attention ════════════════════════════════════════════════════
run_experiment('sparse', 'sinusoidal')

---
## Part 3 — Positional Encodings (all with Vanilla MHA)

In [ ]:
# ══ 3.1  RoPE ════════════════════════════════════════════════════════════════
run_experiment('vanilla', 'rope')

In [ ]:
# ══ 3.2  ALiBi ═══════════════════════════════════════════════════════════════
run_experiment('vanilla', 'alibi')

In [ ]:
# ══ 3.3  Relative Positional Bias ════════════════════════════════════════════
run_experiment('vanilla', 'relative')

---
## Part 4 — Hybrid Models

Change `BEST_ATTN` and `BEST_POS` to whichever performed best in Parts 2 and 3.

In [ ]:
# ← Update these to your best performers from Parts 2 and 3
BEST_ATTN = 'sliding_window'
BEST_POS  = 'alibi'

print(f'Hybrid experiments will use: attn={BEST_ATTN}  pos={BEST_POS}')

In [ ]:
# ══ 4.1  Hybrid: Conv1D Before Each Attention Block ══════════════════════════
run_experiment(
    BEST_ATTN, BEST_POS, model='hybrid',
    extra=['model.hybrid.type=conv_before_attn'],
)

In [ ]:
# ══ 4.2  Hybrid: Gated Convolutional FFN ═════════════════════════════════════
run_experiment(
    BEST_ATTN, BEST_POS, model='hybrid',
    extra=['model.hybrid.type=gated_conv_ffn'],
)

In [ ]:
# ══ 4.3  Hybrid: Interleaved Conv + Attention Blocks ═════════════════════════
run_experiment(
    BEST_ATTN, BEST_POS, model='hybrid',
    extra=['model.hybrid.type=interleaved'],
)

---
## Part 5 — Benchmark

Runs inference latency + perplexity at multiple sequence lengths for every checkpoint.

In [ ]:
# ══ 5  Benchmark all saved checkpoints ═══════════════════════════════════════
import subprocess, sys
REPO = '/content/SAiDL-Summer-Assignment-2026'

# All (attn, pos, model) combos that were trained above
EXPERIMENTS = [
    ('vanilla',        'sinusoidal', 'transformer'),
    ('sliding_window', 'sinusoidal', 'transformer'),
    ('gqa',            'sinusoidal', 'transformer'),
    ('relu',           'sinusoidal', 'transformer'),
    ('sparse',         'sinusoidal', 'transformer'),
    ('vanilla',        'rope',       'transformer'),
    ('vanilla',        'alibi',      'transformer'),
    ('vanilla',        'relative',   'transformer'),
    # Add hybrid entries here if desired
]

for attn, pos, model in EXPERIMENTS:
    print(f'\nBenchmarking: {attn}+{pos}+{model}')
    subprocess.run(
        [
            sys.executable,
            f'{REPO}/core_ml/benchmark.py',
            f'attention={attn}',
            f'positional={pos}',
            f'model={model}',
            'dataset.batch_size=4',
            'dataset.num_workers=2',
        ],
        cwd=REPO,
    )

print('\n✅ All benchmarks done.')

---
## Download Results

In [ ]:
# ══ 6  Zip and download everything ═══════════════════════════════════════════
REPO = '/content/SAiDL-Summer-Assignment-2026'

!cd {REPO} && zip -r /content/saidl_core_ml_results.zip \
    outputs benchmark_results 2>/dev/null || true

from google.colab import files
files.download('/content/saidl_core_ml_results.zip')
print('Download started.')